# LUNAR - multi-run experiments

In [ ]:
import sys
import time
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
import optuna

sys.path.append("../src")
from data import make_optuna_subsample, make_final_subsample, clear_dataset_cache
from optuna_utils import run_study
from metrics import find_best_f1_threshold, minmax_scale_scores, evaluate_scores, print_metrics, compare_threshold_strategies
from results import build_experiment_record, save_record_json

sys.path.append("../external/LUNAR")
import LUNAR
import variables as var

In [ ]:
import shutil

def get_lunar_internal_model_path(dataset, seed, k):
    return Path(f"saved_models/{dataset}/{k}/net_{seed}.pth")

def copy_lunar_model_to_models_dir(dataset, run_index, seed, k):
    source_path = get_lunar_internal_model_path(dataset, seed, k)
    if not source_path.exists():
        raise FileNotFoundError(f"Expected LUNAR checkpoint not found: {source_path}")

    target_dir = MODELS_DIR / dataset
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / f"run_{run_index}_k_{k}_seed_{seed}.pth"
    shutil.copy2(source_path, target_path)
    print(f"Copied model checkpoint to: {target_path}")
    return target_path


In [ ]:
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["CICIDS", "UNSW_NB15"]
SEED = 29
N_TRIALS = 200
SAMPLE_TYPES = ["UNIFORM", "SUBSPACE", "MIXED"]

DATASET_VERSION = "v1"
PREPROCESSING_VERSION = "v1"
SPLIT_METHOD = "stratified_train_val_test_fixed_seed"
MODEL_TYPE = "LUNAR"
FUSION_STRATEGY = "none"

RUN_CONFIGS = [
    dict(run_index=1, n_train_opt=7000,  n_val_opt=3000,
         n_train_final=154000, n_val_final=66000, n_test_final=100000,
         notes="run1_small_opt_sample"),
    dict(run_index=2, n_train_opt=21000, n_val_opt=9000,
         n_train_final=154000, n_val_final=66000, n_test_final=100000,
         notes="run2_medium_opt_sample"),
    dict(run_index=3, n_train_opt=35000, n_val_opt=15000,
         n_train_final=154000, n_val_final=66000, n_test_final=100000,
         notes="run3_large_opt_sample"),
]

In [ ]:
def cleanup_memory():
    gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

In [ ]:
def make_objective(train_x, train_y, val_x, val_y, dataset):
    def objective(trial):
        n_pos = int(val_y.sum())
        n_neg = len(val_y) - n_pos
        if n_pos < 5 or n_neg < 5:
            raise optuna.exceptions.TrialPruned()

        k = trial.suggest_int("k", 5, 150, log=True)
        samples = trial.suggest_categorical("samples", SAMPLE_TYPES)
        lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
        wd = trial.suggest_float("wd", 1e-4, 1.0, log=True)
        epsilon = trial.suggest_float("epsilon", 0.01, 0.5)
        proportion = trial.suggest_int("proportion", 1, 2)
        n_epochs = trial.suggest_int("n_epochs", 50, 300, step=25)

        var.lr, var.wd, var.epsilon = lr, wd, epsilon
        var.proportion, var.n_epochs = proportion, n_epochs

        try:
            out = LUNAR.run(
                train_x, train_y, val_x, val_y, val_x, val_y,
                dataset, SEED, k, samples, train_new_model=True,
            )
            auc = roc_auc_score(val_y, out.numpy())
        except RuntimeError as e:
            if "memory" in str(e).lower():
                gc.collect()
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
                raise optuna.exceptions.TrialPruned()
            raise
        except ValueError:
            raise optuna.exceptions.TrialPruned()

        return auc
    return objective


In [ ]:
def fit_and_score_lunar(params, dataset, train_x, train_y, val_x, val_y, test_x, test_y):
    var.lr = params["lr"]
    var.wd = params["wd"]
    var.epsilon = params["epsilon"]
    var.proportion = params["proportion"]
    var.n_epochs = params["n_epochs"]

    original_device = var.device
    var.device = torch.device("cpu")

    try:
        start_train = time.time()
        out_val = LUNAR.run(
            train_x, train_y, val_x, val_y, val_x, val_y,
            dataset, SEED, params["k"], params["samples"], train_new_model=True,
        )
        runtime_train = time.time() - start_train
        scores_val = minmax_scale_scores(out_val.numpy())

        cleanup_memory()

        start_inference = time.time()
        out_test = LUNAR.run(
            train_x, train_y, val_x, val_y, test_x, test_y,
            dataset, SEED, params["k"], params["samples"], train_new_model=False,
        )
        runtime_inference = time.time() - start_inference
        scores_test = minmax_scale_scores(out_test.numpy())

        return scores_val, scores_test, runtime_train, runtime_inference
    finally:
        var.device = original_device
        cleanup_memory()

In [ ]:
def run_experiment(dataset, run_cfg):
    run_index = run_cfg["run_index"]
    study_name = f"LUNAR_{dataset}_run{run_index}"
    best_params = {}

    train_x, train_y, val_x, val_y = make_optuna_subsample(
        dataset, SEED, run_cfg["n_train_opt"], run_cfg["n_val_opt"]
    )
    objective = make_objective(train_x, train_y, val_x, val_y, dataset)
    study = run_study(objective, study_name, SEED, N_TRIALS, results_dir=RESULTS_DIR)

    best_params = dict(study.best_params)

    del objective, study
    del train_x, train_y, val_x, val_y
    cleanup_memory()

    train_x, train_y, val_x, val_y, test_x, test_y = make_final_subsample(
        dataset, SEED,
        run_cfg["n_train_final"], run_cfg["n_val_final"], run_cfg["n_test_final"],
        max_nodes_budget=50_000_000, k=best_params["k"],
    )

    scores_val, scores_test, runtime_train, runtime_inference = fit_and_score_lunar(
        best_params, dataset, train_x, train_y, val_x, val_y, test_x, test_y
    )

    threshold_candidates = compare_threshold_strategies(val_y, scores_val, beta=2.0, normal_q=0.99)
    chosen_threshold_info = {"selection_method": "validation_f1_max", **threshold_candidates["f1_max"]}
    best_threshold = chosen_threshold_info["threshold"]
    print(f"[{dataset} run{run_index}] threshold candidates: {threshold_candidates}")
    print(f"[{dataset} run{run_index}] chosen threshold={best_threshold:.4f} by validation_f1_max")

    metrics = evaluate_scores(test_y, scores_test, threshold=best_threshold)
    print_metrics(f"LUNAR final - {dataset} run{run_index}", metrics)

    copied_model_path = copy_lunar_model_to_models_dir(dataset, run_index, SEED, best_params["k"])

    record = build_experiment_record(
        dataset_name=dataset,
        dataset_version=DATASET_VERSION,
        split_method=SPLIT_METHOD,
        seed=SEED,
        preprocessing_version=PREPROCESSING_VERSION,
        model_type=MODEL_TYPE,
        fusion_strategy=FUSION_STRATEGY,
        hyperparameters=best_params,
        threshold=best_threshold,
        scores_test=scores_test,
        test_y=test_y,
        runtime_train=runtime_train,
        runtime_inference=runtime_inference,
        threshold_info={"candidates": threshold_candidates, "chosen": chosen_threshold_info},
        model_path=copied_model_path,
    )
    save_record_json(record, RESULTS_DIR, run_index, MODEL_TYPE, dataset)
    del train_x, train_y, val_x, val_y, test_x, test_y, scores_val, scores_test
    cleanup_memory()

    return record



In [ ]:
all_records = []

In [ ]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[0])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[0])
all_records.append(rec)
pd.DataFrame(all_records)
clear_dataset_cache()

In [ ]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[1])
all_records.append(rec)
pd.DataFrame(all_records)
clear_dataset_cache()

In [ ]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[1])
all_records.append(rec)
pd.DataFrame(all_records)
clear_dataset_cache()

In [ ]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[2])
all_records.append(rec)
pd.DataFrame(all_records)
clear_dataset_cache()

In [ ]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[2])
all_records.append(rec)
pd.DataFrame(all_records)
clear_dataset_cache()